In [11]:
!pip install git+https://github.com/stanford-futuredata/ColBERT.git@main torch==2.1.2 langdetect==1.0.9


  Cloning https://github.com/stanford-futuredata/ColBERT.git (to revision main) to /private/var/folders/f5/pgcxx91x31l672vfgh983wcr0000gn/T/pip-req-build-ecbcv35m
  Running command git clone --filter=blob:none --quiet https://github.com/stanford-futuredata/ColBERT.git /private/var/folders/f5/pgcxx91x31l672vfgh983wcr0000gn/T/pip-req-build-ecbcv35m
  Resolved https://github.com/stanford-futuredata/ColBERT.git to commit 8d5efcd178f31862a9dbcbf675693a3ed2e9bcc6
  Preparing metadata (setup.py) ... done
ERROR: Could not find a version that satisfies the requirement faiss-gpu==1.7.2 (from versions: none)
ERROR: No matching distribution found for faiss-gpu==1.7.2


In [14]:
from colbert.infra import Run, RunConfig, ColBERTConfig
from colbert.data import Queries, Collection
from colbert import Indexer, Searcher

In [ ]:

n_gpu: int = 0 # Set your number of available GPUs
experiment: str = "colbert" # Name of the folder where the logs and created indices will be stored
index_name: str = "my_index" # The name of your index, i.e. the name of your vector database
documents: list = ["Ceci est un premier document.", "Voici un second document.", "etc."] # Corpus

# Step 1: Indexing. This step encodes all passages into matrices, stores them on disk, and builds data structures for efficient search.
with Run().context(RunConfig(nranks=n_gpu,experiment=experiment)):
    indexer = Indexer(checkpoint="antoinelouis/colbert-xm")
    indexer.index(name=index_name, collection=documents)

# Step 2: Searching. Given the model and index, you can issue queries over the collection to retrieve the top-k passages for each query.
with Run().context(RunConfig(nranks=n_gpu,experiment=experiment)):
    searcher = Searcher(index=index_name) # You don't need to specify checkpoint again, the model name is stored in the index.
    results = searcher.search(query="Comment effectuer une recherche avec ColBERT ?", k=10)
    # results: tuple of tuples of length k containing ((passage_id, passage_rank, passage_score), ...)
